# Notebook 02 - Function App Por Usuario

Despliega endpoints de negocio por participante en su RG.

### Descripcion de la celda 1
Carga configuracion, valida variables y prepara el contexto.

In [27]:
# Celda 2: ejecuta este paso del notebook y prepara resultados para el siguiente.

import json
import os
import shutil
import subprocess
import textwrap
import zipfile
from pathlib import Path

def resolve_workdir() -> Path:
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / 'config.env').exists():
            return c
    return cwd

def resolve_az_cli() -> str:
    for candidate in ('az', 'az.cmd', 'az.exe'):
        found = shutil.which(candidate)
        if found:
            return found
    raise RuntimeError(
        'No se encontro Azure CLI en el PATH del kernel. '
        'En Windows instala Azure CLI y reinicia VS Code/Jupyter.'
    )

WORKDIR = resolve_workdir()
AZ_CLI = resolve_az_cli()

def load_env_file(path='config.env'):
    env_map = {}
    p = Path(path)
    if not p.exists():
        return env_map
    for raw in p.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        env_map[k.strip()] = v.strip().strip("\"").strip("'")
    return env_map

env_file = load_env_file(WORKDIR / 'config.env')
cfg = {}
cfg_path = WORKDIR / 'workshop_config.json'
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))

def env(name, default=None, required=False):
    v = os.getenv(name, env_file.get(name, default))
    if required and (v is None or str(v).strip() == ''):
        raise RuntimeError(f'Falta variable requerida: {name}')
    return v

SUBSCRIPTION_ID = env('AZURE_SUBSCRIPTION_ID', cfg.get('azure_subscription_id'), required=True)
LOCATION = env('AZURE_LOCATION', cfg.get('azure_location', 'eastus'))
USER_ALIAS = env('USER_ALIAS', cfg.get('user_alias', 'user01')).strip()
USER_RG_NAME = env('USER_RG_NAME', cfg.get('user_rg_name', f'rg-foundry-demo-{USER_ALIAS}'))
RESOURCE_PREFIX = env('RESOURCE_PREFIX', cfg.get('resource_prefix', f'contoso-{USER_ALIAS}'))

cfg = {
    **cfg,
    'azure_openai_endpoint': env('AZURE_OPENAI_ENDPOINT', cfg.get('azure_openai_endpoint', '')),
    'api_version': env('AZURE_OPENAI_API_VERSION', env('API_VERSION', cfg.get('api_version', '2025-01-01-preview')))
}

FUNCTION_APP_NAME = f"{USER_ALIAS}contosofunc".replace('-', '')[:60]
STORAGE_ACCOUNT_NAME = f"{USER_ALIAS}funcstg".replace('-', '')[:24]

def run(cmd):
    final_cmd = cmd.copy()
    if final_cmd and final_cmd[0] == 'az':
        final_cmd[0] = AZ_CLI
    p = subprocess.run(final_cmd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr or p.stdout)
    return p.stdout.strip()

run(['az', 'account', 'set', '--subscription', SUBSCRIPTION_ID])
print('Workdir:', WORKDIR)
print('Azure CLI:', AZ_CLI)
print('Function App:', FUNCTION_APP_NAME)

Workdir: /Users/williamtalero/Documents/Proyectos/Microsoft/WorkShop - Agentes
Azure CLI: /opt/homebrew/bin/az
Function App: user01contosofunc


### Descripcion de la celda 2
Provisiona y despliega la Function App con funciones de negocio.

In [ ]:
# Celda 3: ejecuta este paso del notebook y prepara resultados para el siguiente.

import time

stg_probe = subprocess.run([
    AZ_CLI, 'storage', 'account', 'show',
    '--name', STORAGE_ACCOUNT_NAME,
    '--resource-group', USER_RG_NAME
], capture_output=True, text=True)
if stg_probe.returncode != 0:
    run([
        'az', 'storage', 'account', 'create',
        '--name', STORAGE_ACCOUNT_NAME,
        '--resource-group', USER_RG_NAME,
        '--location', LOCATION,
        '--sku', 'Standard_LRS'
    ])
    print('Storage creado')
else:
    print('Storage ya existia')

func_probe = subprocess.run([
    AZ_CLI, 'functionapp', 'show',
    '--name', FUNCTION_APP_NAME,
    '--resource-group', USER_RG_NAME
], capture_output=True, text=True)
if func_probe.returncode != 0:
    run([
        'az', 'functionapp', 'create',
        '--resource-group', USER_RG_NAME,
        '--name', FUNCTION_APP_NAME,
        '--storage-account', STORAGE_ACCOUNT_NAME,
        '--runtime', 'python',
        '--runtime-version', '3.11',
        '--functions-version', '4',
        '--flexconsumption-location', LOCATION
    ])
    print('Function App creada (Flex Consumption)')
else:
    print('Function App ya existia')

# Configuracion de deployment storage por identidad para entornos donde key auth esta bloqueada
run(['az', 'functionapp', 'identity', 'assign', '--resource-group', USER_RG_NAME, '--name', FUNCTION_APP_NAME, '-o', 'none'])
principal_id = run(['az', 'functionapp', 'identity', 'show', '--resource-group', USER_RG_NAME, '--name', FUNCTION_APP_NAME, '--query', 'principalId', '-o', 'tsv'])
storage_id = run(['az', 'storage', 'account', 'show', '--resource-group', USER_RG_NAME, '--name', STORAGE_ACCOUNT_NAME, '--query', 'id', '-o', 'tsv'])

ra_probe = subprocess.run([
    AZ_CLI, 'role', 'assignment', 'list',
    '--assignee', principal_id,
    '--scope', storage_id,
    '--query', "[?roleDefinitionName=='Storage Blob Data Contributor'] | length(@)",
    '-o', 'tsv'
], capture_output=True, text=True)
if ra_probe.returncode != 0 or ra_probe.stdout.strip() == '0':
    run([
        'az', 'role', 'assignment', 'create',
        '--assignee-object-id', principal_id,
        '--assignee-principal-type', 'ServicePrincipal',
        '--role', 'Storage Blob Data Contributor',
        '--scope', storage_id,
        '-o', 'none'
    ])
    print('Rol Storage Blob Data Contributor asignado a la Function App')
else:
    print('Rol Storage Blob Data Contributor ya existia')

container_name = f"app-package-{RESOURCE_PREFIX}"[:63]
try:
    run([
        'az', 'storage', 'container', 'create',
        '--name', container_name,
        '--account-name', STORAGE_ACCOUNT_NAME,
        '--auth-mode', 'login',
        '-o', 'none'
    ])
except RuntimeError as ex:
    msg = str(ex).lower()
    if 'network rules' in msg or 'blocked by network rules' in msg:
        run([
            'az', 'storage', 'container-rm', 'create',
            '--storage-account', STORAGE_ACCOUNT_NAME,
            '--resource-group', USER_RG_NAME,
            '--name', container_name,
            '-o', 'none'
        ])
        print('Contenedor creado via management plane por restricciones de red en data plane.')
    else:
        raise

run([
    'az', 'functionapp', 'deployment', 'config', 'set',
    '--resource-group', USER_RG_NAME,
    '--name', FUNCTION_APP_NAME,
    '--deployment-storage-name', STORAGE_ACCOUNT_NAME,
    '--deployment-storage-container-name', container_name,
    '--deployment-storage-auth-type', 'SystemAssignedIdentity',
    '-o', 'none'
])

run(['az', 'functionapp', 'restart', '--resource-group', USER_RG_NAME, '--name', FUNCTION_APP_NAME, '-o', 'none'])
print('Infra de Function App lista')

Proveedor Microsoft.Web ya registrado
Storage ya existia
Function App ya existia
Rol Storage Blob Data Contributor ya existia
Contenedor creado via management plane por restricciones de red en data plane.
Infra de Function App lista


### Descripcion de la celda 3
Provisiona y despliega la Function App con funciones de negocio.

In [29]:
# Celda 4: ejecuta este paso del notebook y prepara resultados para el siguiente.

app_dir = Path('functionapps') / RESOURCE_PREFIX
app_dir.mkdir(parents=True, exist_ok=True)

(app_dir / 'requirements.txt').write_text('azure-functions\n', encoding='utf-8')
(app_dir / 'host.json').write_text(json.dumps({'version': '2.0'}, indent=2), encoding='utf-8')

for src, dst in [
    ('data/clientes.json', 'clientes.json'),
    ('data/productos_crediticios.json', 'productos_crediticios.json'),
    ('data/tickets.json', 'tickets.json')
]:
    (app_dir / dst).write_text(Path(src).read_text(encoding='utf-8'), encoding='utf-8')

function_code = textwrap.dedent('''
import azure.functions as func
import json
from pathlib import Path

app = func.FunctionApp(http_auth_level=func.AuthLevel.ANONYMOUS)
BASE = Path(__file__).parent

CLIENTES = json.loads((BASE / 'clientes.json').read_text(encoding='utf-8'))
PRODUCTOS = json.loads((BASE / 'productos_crediticios.json').read_text(encoding='utf-8'))
TICKETS = json.loads((BASE / 'tickets.json').read_text(encoding='utf-8'))

@app.route(route='buscar_cliente', methods=['POST'])
def buscar_cliente(req: func.HttpRequest) -> func.HttpResponse:
    body = req.get_json()
    cedula = body.get('cedula')
    for c in CLIENTES:
        if c.get('cedula') == cedula:
            return func.HttpResponse(json.dumps(c, ensure_ascii=False), mimetype='application/json')
    return func.HttpResponse(json.dumps({'error': 'no encontrado'}, ensure_ascii=False), mimetype='application/json', status_code=404)

@app.route(route='listar_productos_disponibles', methods=['POST'])
def listar_productos_disponibles(req: func.HttpRequest) -> func.HttpResponse:
    return func.HttpResponse(json.dumps(PRODUCTOS, ensure_ascii=False), mimetype='application/json')

@app.route(route='consultar_tickets_cliente', methods=['POST'])
def consultar_tickets_cliente(req: func.HttpRequest) -> func.HttpResponse:
    body = req.get_json()
    cliente_id = body.get('cliente_id')
    out = [t for t in TICKETS if t.get('cliente_id') == cliente_id]
    return func.HttpResponse(json.dumps({'tickets': out}, ensure_ascii=False), mimetype='application/json')
''').strip() + '\n'

(app_dir / 'function_app.py').write_text(function_code, encoding='utf-8')
print('Codigo de funciones generado')

Codigo de funciones generado


### Descripcion de la celda 4
Provisiona y despliega la Function App con funciones de negocio.

In [ ]:
# Celda 5: ejecuta este paso del notebook y prepara resultados para el siguiente.

zip_path = app_dir / 'deploy.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in app_dir.glob('*'):
        if file_path.name != 'deploy.zip':
            zf.write(file_path, arcname=file_path.name)

deploy = subprocess.run([
    AZ_CLI, 'functionapp', 'deployment', 'source', 'config-zip',
    '--resource-group', USER_RG_NAME,
    '--name', FUNCTION_APP_NAME,
    '--src', str(zip_path)
], capture_output=True, text=True)
combined = (deploy.stdout or '') + '\n' + (deploy.stderr or '')

default_host = run([
    'az', 'functionapp', 'show',
    '--resource-group', USER_RG_NAME,
    '--name', FUNCTION_APP_NAME,
    '--query', 'defaultHostName',
    '-o', 'tsv'
]).strip()
if not default_host:
    default_host = f"{FUNCTION_APP_NAME}.azurewebsites.net"
function_base_url = f"https://{default_host}/api"

published = run([
    'az', 'functionapp', 'function', 'list',
    '--resource-group', USER_RG_NAME,
    '--name', FUNCTION_APP_NAME,
    '--query', 'length(@)',
    '-o', 'tsv'
])
if int(published or '0') == 0:
    raise RuntimeError(f'Deploy no publico funciones. Salida CLI:\n{combined}')

if deploy.returncode != 0:
    print('Aviso deploy CLI (se continua porque hay funciones publicadas):')
    print(combined)

print('Function App desplegada:', function_base_url)
print('Funciones publicadas:', published)

### Descripcion de la celda 5
Actualiza y guarda el estado para los siguientes notebooks.

In [ ]:
# Celda 6: ejecuta este paso del notebook y prepara resultados para el siguiente.

if 'function_base_url' not in globals():
    raise RuntimeError('Function App no desplegada. Corrige celdas previas antes de finalizar.')

FUNCTION_APP_CONFIG = {
    'name': FUNCTION_APP_NAME,
    'base_url': function_base_url
}

state = {}
config_path = Path('workshop_config.json')
if config_path.exists():
    state = json.loads(config_path.read_text(encoding='utf-8'))

state.update({
    'azure_subscription_id': SUBSCRIPTION_ID,
    'azure_location': LOCATION,
    'user_alias': USER_ALIAS,
    'user_rg_name': USER_RG_NAME,
    'resource_prefix': RESOURCE_PREFIX,
    'function_app': FUNCTION_APP_CONFIG,
    'function_app_ready': True
})
config_path.write_text(json.dumps(state, indent=2), encoding='utf-8')

print('OK Function App configurada')
print('Estado actualizado: workshop_config.json')
print(json.dumps(FUNCTION_APP_CONFIG, indent=2))

OK Function App configurada
